<a href="https://colab.research.google.com/github/SujahathMSM/PdfToLinkedInPosts/blob/dev/LinkedInPostGenusinfGEMINIAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install google-generativeai diffusers torch torchvision ipywidgets matplotlib PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 68.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12

In [3]:
import google.generativeai as genai
from google.colab import files
import PyPDF2
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, Button, VBox, Output

# Configure the Gemini API key
genai.configure(api_key="Enter your GEMINI API KEY")  # Replace with your Gemini API key

# Select the Gemini model
model = genai.GenerativeModel('gemini-pro')

# Upload PDF file
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]  # Get the uploaded file name

# Text extraction from the PDF
def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        num_pages = len(reader.pages)
        print(f"PDF has {num_pages} pages. Extracting text...")
        for i, page in enumerate(reader.pages):
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
            if (i + 1) % 100 == 0:
                print(f"Processed {i + 1} pages...")
    return text

# Split text into chunks
def chunk_text(text, chunk_size=5000):
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

# Summarize a chunk of text using Gemini
def summarize_text(text_chunk):
    prompt = f"Summarize the following text concisely:\n\n{text_chunk}"
    try:
        response = model.generate_content(prompt)
        summary = response.text.strip()
        return summary
    except Exception as e:
        print("Error during summarization:", e)
        return ""

# Aggregate summaries from all text chunks
def aggregate_summaries(pdf_text, chunk_size=5000):
    print("Splitting PDF text into chunks...")
    chunks = chunk_text(pdf_text, chunk_size)
    summaries = []
    print(f"Total chunks to summarize: {len(chunks)}")
    for idx, chunk in enumerate(chunks):
        print(f"Summarizing chunk {idx + 1}/{len(chunks)}...")
        summary = summarize_text(chunk)
        summaries.append(summary)
    aggregated_summary = "\n".join(summaries)
    return aggregated_summary

# Generate post ideas using Gemini
def generate_post_ideas(aggregated_summary):
    prompt = (
        "Based on the following summarized content, generate 10 unique, creative, and professional ideas "
        "for a LinkedIn post. Each idea should be concise:\n\n"
        f"{aggregated_summary}\n\n"
        "List the ideas, numbered 1 to 10:"
    )
    try:
        response = model.generate_content(prompt)
        ideas = response.text.strip()
        return ideas
    except Exception as e:
        print("Error generating post ideas:", e)
        return ""

# Generate a full LinkedIn post from a selected idea and aggregated summary using Gemini
def generate_linkedin_post(selected_idea, aggregated_summary):
    prompt = (
        f"Using the idea: '{selected_idea}', and considering the following context:\n\n"
        f"{aggregated_summary}\n\n"
        "Write a detailed, engaging, and professional LinkedIn post."
    )
    try:
        response = model.generate_content(prompt)
        linkedin_post = response.text.strip()
        return linkedin_post
    except Exception as e:
        print("Error generating LinkedIn post:", e)
        return ""

# Generate AI image using Stable Diffusion via diffusers
def generate_ai_image(linkedin_post):
    # You can customize the image prompt based on the LinkedIn post if desired.
    image_prompt = (
        "Generate a high-resolution, modern digital illustration that conveys a professional and innovative atmosphere. "
        "Incorporate a clean, minimalistic design with vibrant colors and dynamic composition. "
        "The illustration should evoke creativity and success, suitable for a professional setting. "
        "Do not include any text or lettering in the image."
    )

    try:
        import torch
        from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

        torch.cuda.empty_cache()
        model_id = "stabilityai/stable-diffusion-2-1"

        pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
        pipe = pipe.to("cuda")

        # Generate image with desired dimensions (adjust width/height as needed)
        image = pipe(image_prompt, width=1024, height=1024).images[0]
        return image  # This is a PIL Image object
    except Exception as e:
        print("Error during image generation:", e)
        return None

# Main Execution
pdf_text = extract_text_from_pdf(pdf_path)

print("Aggregating summaries from PDF text...")
aggregated_summary = aggregate_summaries(pdf_text, chunk_size=5000)
print("Aggregated Summary Generated.\n")

print("Generating LinkedIn post ideas...")
ideas_text = generate_post_ideas(aggregated_summary)
print("Post Ideas Generated:\n", ideas_text)

ideas_list = []
for line in ideas_text.split("\n"):
    line = line.strip()
    if line and (line[0].isdigit() or line.startswith("-")):
        idea = line.lstrip("0123456789.- ").strip()
        ideas_list.append(idea)

dropdown = Dropdown(options=ideas_list, description='Select Idea:')
button = Button(description="Generate LinkedIn Post")
output = Output()

def on_button_click(b):
    with output:
        output.clear_output()
        selected_idea = dropdown.value
        print("Selected Idea:", selected_idea, "\n")
        print("Generating LinkedIn post... please wait.")
        linkedin_post = generate_linkedin_post(selected_idea, aggregated_summary)
        print("\nGenerated LinkedIn Post:")
        print(linkedin_post)

        print("\nGenerating AI image... please wait.")
        # Generate the image using Stable Diffusion
        image = generate_ai_image(linkedin_post)
        if image:
            print("\nDisplaying generated AI image...")
            plt.figure(figsize=(8, 6))
            plt.imshow(image)
            plt.axis('off')
            plt.show()
        else:
            print("\nNo image was generated.")

button.on_click(on_button_click)
display(VBox([dropdown, button, output]))


Saving How to Improve Your Concentration The 7 Secrets of How to Improve Your Memory and to Stay Focused by Thompson, Oliver (z-lib.org).pdf to How to Improve Your Concentration The 7 Secrets of How to Improve Your Memory and to Stay Focused by Thompson, Oliver (z-lib.org).pdf
PDF has 43 pages. Extracting text...
Aggregating summaries from PDF text...
Splitting PDF text into chunks...
Total chunks to summarize: 7
Summarizing chunk 1/7...
Summarizing chunk 2/7...
Summarizing chunk 3/7...
Summarizing chunk 4/7...
Summarizing chunk 5/7...
Summarizing chunk 6/7...
Summarizing chunk 7/7...
Aggregated Summary Generated.

Generating LinkedIn post ideas...
Post Ideas Generated:
 1. Unleash Your Focus: Discover the Brain's Hidden Pool of Concentration
2. The Art of Concentration: Master Your Surroundings for Optimal Focus
3. Unraveling the Science of Concentration: The Role of Lateral Intra-parietal Cortex
4. Image vs. Tract: How Concentration Shapes Your Memory and Learning
5. Dehydration's Hi